# 06 - Feature Engineering

## Objective

This notebook transforms the cleaned 2025 Airline On-Time Performance dataset into model-ready features for schedule-time flight-delay prediction.

Only information available before the scheduled departure of a flight will be used as predictive input. Variables generated during or after flight operations are excluded to prevent target leakage.

The feature-engineering process includes:

- Loading and validating the cleaned Delta table
- Creating the model-eligible flight population
- Engineering temporal features
- Engineering schedule features
- Selecting the final predictive variables
- Validating the engineered dataset
- Saving the feature dataset for model training


#### Load configuration and cleaned dataset

The feature-engineering process begins by loading the managed `flights_clean` Delta table produced by the data-cleaning notebook.

The source table is validated before transformations are applied. The raw and cleaned datasets remain unchanged throughout this notebook.


In [1]:
from __future__ import annotations

import importlib.util
from pathlib import Path

_bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not _bootstrap.is_file():
    _bootstrap = Path.cwd() / "import_path.py"
_spec = importlib.util.spec_from_file_location("import_path", _bootstrap)
_ip = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_ip)

# Load the project configuration

from config import project_config as cfg
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Project configuration loaded successfully.")
print(f"Source table: {cfg.CLEAN_TABLE}")
print(f"Output table: {cfg.FEATURES_TABLE}")
print(f"Prediction target: {cfg.TARGET_COLUMN}")


Project configuration loaded successfully.
Source table: workspace.default.flights_clean
Output table: workspace.default.flights_features
Prediction target: ARR_DEL15


In [2]:
# Load and validate the cleaned dataset

def require_table(table_name: str) -> None:
    """Raise an error when a required Unity Catalog table is unavailable."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the data-cleaning notebook before continuing."
        )


require_table(cfg.CLEAN_TABLE)

df_clean: DataFrame = spark.table(cfg.CLEAN_TABLE)

clean_row_count = df_clean.count()
clean_column_count = len(df_clean.columns)

missing_columns = sorted(set(cfg.FEATURE_INPUT_COLUMNS) - set(df_clean.columns))

if missing_columns:
    raise ValueError(
        "Feature-engineering validation failed. "
        f"Missing required columns: {missing_columns}"
    )

print("Cleaned dataset loaded and validated successfully.")
print(f"Source table: {cfg.CLEAN_TABLE}")
print(f"Output table: {cfg.FEATURES_TABLE}")
print(f"Total records: {clean_row_count:,}")
print(f"Total columns: {clean_column_count}")
print(f"Prediction target: {cfg.TARGET_COLUMN}")


Cleaned dataset loaded and validated successfully.
Source table: workspace.default.flights_clean
Output table: workspace.default.flights_features
Total records: 7,001,611
Total columns: 32
Prediction target: ARR_DEL15


#### Create model-eligible dataset

The cleaned dataset contains all historical flight records, including cancelled and diverted flights.

For supervised machine learning, only flights that completed normal arrival operations are retained. Cancelled and diverted flights are excluded because the prediction target represents arrival delay for completed flights.


In [3]:
# Create model-eligible dataset

df_model = (
    df_clean
    .filter(
        (F.col(cfg.CANCELLED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
        & (F.col(cfg.DIVERTED_COLUMN) == cfg.MODEL_ELIGIBLE_STATUS_VALUE)
    )
)

model_row_count = df_model.count()
removed_records = clean_row_count - model_row_count

model_summary = spark.createDataFrame(
    [
        (
            clean_row_count,
            model_row_count,
            removed_records,
        )
    ],
    schema="CLEAN_DATASET_ROWS long, MODEL_DATASET_ROWS long, EXCLUDED_RECORDS long",
)

display(model_summary)


,CLEAN_DATASET_ROWS,MODEL_DATASET_ROWS,EXCLUDED_RECORDS
0,7001611,6879483,122128


#### Engineer temporal features

Temporal features capture calendar- and time-related patterns that may influence flight delays. These variables are derived exclusively from information available before the scheduled departure of a flight.

The following temporal features are created:

- `DEP_HOUR` — scheduled departure hour extracted from `CRS_DEP_TIME`
- `DEP_MINUTE` — scheduled departure minute extracted from `CRS_DEP_TIME`
- `ARR_HOUR` — scheduled arrival hour extracted from `CRS_ARR_TIME`
- `ARR_MINUTE` — scheduled arrival minute extracted from `CRS_ARR_TIME`
- `IS_WEEKEND` — indicates whether the scheduled flight departs on a weekend
- `SEASON` — meteorological season derived from the flight month

`CRS_DEP_TIME` and `CRS_ARR_TIME` are stored in HHMM format, which is not an ordinary continuous integer (for example, the gap from 23:59 to 00:01 is 2 real minutes but a ~2358-unit numeric jump). Decomposing both into hour and minute avoids that distortion; the raw HHMM columns are kept in the feature table for reporting purposes only and are excluded from the model's numeric inputs.

In [4]:
# Engineer temporal features

df_features = df_model

# CRS_DEP_TIME / CRS_ARR_TIME occasionally use "2400" for a scheduled
# midnight time. floor(.../100) alone would produce an invalid hour of 24,
# so the result is taken modulo 24 to normalize midnight to hour 0.

df_features = df_features.withColumn(
    cfg.DEP_HOUR_COLUMN,
    (F.floor(F.col(cfg.SCHEDULED_DEPARTURE_COLUMN) / 100) % 24).cast("int"),
)

df_features = df_features.withColumn(
    cfg.DEP_MINUTE_COLUMN,
    (F.col(cfg.SCHEDULED_DEPARTURE_COLUMN) % 100).cast("int"),
)

df_features = df_features.withColumn(
    cfg.ARR_HOUR_COLUMN,
    (F.floor(F.col(cfg.SCHEDULED_ARRIVAL_COLUMN) / 100) % 24).cast("int"),
)

df_features = df_features.withColumn(
    cfg.ARR_MINUTE_COLUMN,
    (F.col(cfg.SCHEDULED_ARRIVAL_COLUMN) % 100).cast("int"),
)

df_features = df_features.withColumn(
    cfg.IS_WEEKEND_COLUMN,
    F.when(
        F.col(cfg.DAY_OF_WEEK_COLUMN).isin(*cfg.WEEKEND_DAYS),
        F.lit(1),
    ).otherwise(F.lit(0)),
)

# Build the SEASON expression from cfg.SEASON_MONTH_GROUPS (single source of
# truth shared with the descriptive-analytics and statistical-analysis notebooks)

season_expression = None
for season_name, season_months in cfg.SEASON_MONTH_GROUPS.items():
    month_condition = F.col(cfg.MONTH_COLUMN).isin(*season_months)
    if season_expression is None:
        season_expression = F.when(month_condition, season_name)
    else:
        season_expression = season_expression.when(month_condition, season_name)

df_features = df_features.withColumn(cfg.SEASON_COLUMN, season_expression)

print("Temporal features created successfully.")
print(f"Rows retained: {df_features.count():,}")
print(f"Columns after temporal engineering: {len(df_features.columns)}")


Temporal features created successfully.
Rows retained: 6,879,483
Columns after temporal engineering: 38


In [5]:
# Preview engineered temporal features

display(
    df_features.select(
        cfg.FLIGHT_DATE_COLUMN,
        cfg.MONTH_COLUMN,
        cfg.DAY_OF_WEEK_COLUMN,
        cfg.SCHEDULED_DEPARTURE_COLUMN,
        cfg.DEP_HOUR_COLUMN,
        cfg.DEP_MINUTE_COLUMN,
        cfg.SCHEDULED_ARRIVAL_COLUMN,
        cfg.ARR_HOUR_COLUMN,
        cfg.ARR_MINUTE_COLUMN,
        cfg.IS_WEEKEND_COLUMN,
        cfg.SEASON_COLUMN,
    ).limit(20)
)


,FL_DATE,MONTH,DAY_OF_WEEK,CRS_DEP_TIME,DEP_HOUR,DEP_MINUTE,CRS_ARR_TIME,ARR_HOUR,ARR_MINUTE,IS_WEEKEND,SEASON
0,2025-04-07,4,1,700,7,0,1020,10,20,0,Spring
1,2025-04-07,4,1,2200,22,0,626,6,26,0,Spring
2,2025-04-07,4,1,620,6,20,935,9,35,0,Spring
3,2025-04-07,4,1,1615,16,15,1745,17,45,0,Spring
4,2025-04-07,4,1,1837,18,37,2154,21,54,0,Spring
5,2025-04-07,4,1,2029,20,29,2124,21,24,0,Spring
6,2025-04-07,4,1,2305,23,5,514,5,14,0,Spring
7,2025-04-07,4,1,2040,20,40,2228,22,28,0,Spring
8,2025-04-07,4,1,1742,17,42,2200,22,0,0,Spring
9,2025-04-07,4,1,1654,16,54,1915,19,15,0,Spring


In [6]:
# Summarize temporal feature distributions

display(
    df_features
    .groupBy(cfg.SEASON_COLUMN)
    .count()
    .orderBy(cfg.SEASON_COLUMN)
)

display(
    df_features
    .groupBy(cfg.IS_WEEKEND_COLUMN)
    .count()
    .orderBy(cfg.IS_WEEKEND_COLUMN)
)

display(
    df_features
    .groupBy(cfg.DEP_HOUR_COLUMN)
    .count()
    .orderBy(cfg.DEP_HOUR_COLUMN)
)


,SEASON,count
0,Fall,1715193
1,Spring,1767605
2,Summer,1806016
3,Winter,1590669


,IS_WEEKEND,count
0,0,4962141
1,1,1917342


,DEP_HOUR,count
0,0,11076
1,1,3393
2,2,1299
3,3,732
4,4,298
5,5,194931
6,6,474704
7,7,501217
8,8,478909
9,9,390739


#### Engineer schedule features

Schedule-related features describe operational characteristics that are known before departure.

The following schedule features are engineered:

- `TIME_OF_DAY` — categorizes scheduled departures into operational periods
- `FLIGHT_DISTANCE_CATEGORY` — groups flights into short-, medium-, and long-haul categories based on scheduled distance


In [7]:
# Engineer schedule features

# Build the TIME_OF_DAY expression from cfg.TIME_OF_DAY_BUCKETS (single source
# of truth shared with the descriptive-analytics and statistical-analysis notebooks)

time_of_day_expression = None
for bucket in cfg.TIME_OF_DAY_BUCKETS:
    hour_condition = F.col(cfg.DEP_HOUR_COLUMN).between(
        bucket["start_hour"], bucket["end_hour"]
    )
    if time_of_day_expression is None:
        time_of_day_expression = F.when(hour_condition, bucket["label"])
    else:
        time_of_day_expression = time_of_day_expression.when(
            hour_condition, bucket["label"]
        )

df_features = df_features.withColumn(
    cfg.TIME_OF_DAY_COLUMN,
    time_of_day_expression,
)

df_features = df_features.withColumn(
    cfg.FLIGHT_DISTANCE_CATEGORY_COLUMN,
    F.when(F.col(cfg.DISTANCE_COLUMN) < cfg.DISTANCE_SHORT_MAX_MILES, "Short")
    .when(
        F.col(cfg.DISTANCE_COLUMN) <= cfg.DISTANCE_MEDIUM_MAX_MILES,
        "Medium",
    )
    .otherwise("Long"),
)

print("Schedule features created successfully.")
print(f"Current columns: {len(df_features.columns)}")


Schedule features created successfully.
Current columns: 40


In [8]:
# Summarize schedule feature distributions

display(
    df_features.groupBy(cfg.TIME_OF_DAY_COLUMN)
    .count()
    .orderBy(cfg.TIME_OF_DAY_COLUMN)
)

display(
    df_features.groupBy(cfg.FLIGHT_DISTANCE_CATEGORY_COLUMN)
    .count()
    .orderBy(cfg.FLIGHT_DISTANCE_CATEGORY_COLUMN)
)


,TIME_OF_DAY,count
0,Afternoon,2006366
1,Evening,1518651
2,Morning,2704876
3,Night,437861
4,Overnight,211729


,FLIGHT_DISTANCE_CATEGORY,count
0,Long,927847
1,Medium,3636126
2,Short,2315510


#### Select final predictive variables

Following the project's target leakage restrictions, only variables available before the scheduled departure of a flight are retained for machine learning.

The final predictive dataset includes calendar, airline, route, geographic, schedule, engineered temporal, engineered schedule, and target variables.


In [9]:
# Select final predictive variables

df_ml = df_features.select(*cfg.MODEL_FEATURE_COLUMNS)

print("Final predictive dataset created successfully.")
print(f"Rows: {df_ml.count():,}")
print(f"Columns: {len(df_ml.columns)}")


Final predictive dataset created successfully.
Rows: 6,879,483
Columns: 25


In [10]:
# List final predictive variables

print("Final Predictive Variables")

for column in df_ml.columns:
    print(column)


Final Predictive Variables
QUARTER
MONTH
DAY_OF_WEEK
FL_DATE
OP_UNIQUE_CARRIER
OP_CARRIER_FL_NUM
ORIGIN
DEST
DISTANCE
ORIGIN_CITY_NAME
ORIGIN_STATE_NM
DEST_CITY_NAME
DEST_STATE_NM
CRS_DEP_TIME
CRS_ARR_TIME
CRS_ELAPSED_TIME
DEP_HOUR
DEP_MINUTE
ARR_HOUR
ARR_MINUTE
IS_WEEKEND
SEASON
TIME_OF_DAY
FLIGHT_DISTANCE_CATEGORY
ARR_DEL15


#### Validate engineered dataset

The engineered dataset is validated before being used for machine learning model development.

The validation confirms total record count, feature count, missing values, and data types for the selected predictive variables.


In [11]:
# Summarize engineered dataset dimensions

validation_summary = spark.createDataFrame(
    [
        (
            df_ml.count(),
            len(df_ml.columns),
        )
    ],
    schema="TOTAL_RECORDS long, TOTAL_FEATURES long",
)

display(validation_summary)


,TOTAL_RECORDS,TOTAL_FEATURES
0,6879483,25


In [12]:
# Summarize missing values in final features and stop if any are found

total_rows = df_ml.count()

null_count_expressions = [
    F.sum(F.col(column_name).isNull().cast("long")).alias(column_name)
    for column_name in df_ml.columns
]
null_counts_row = df_ml.select(*null_count_expressions).first()

null_summary = []
for column_name in df_ml.columns:
    null_count = int(null_counts_row[column_name])
    null_percentage = round((null_count / total_rows) * 100, 4)
    null_summary.append((column_name, null_count, null_percentage))

null_summary_df = spark.createDataFrame(
    null_summary,
    ["COLUMN", "NULL_COUNT", "NULL_PERCENTAGE"],
)

display(null_summary_df.orderBy(F.col("NULL_COUNT").desc()))

columns_with_nulls = [
    column_name
    for column_name, null_count, _ in null_summary
    if null_count > 0
]

if columns_with_nulls:
    raise ValueError(
        "The final feature dataset contains missing values in "
        f"columns that should always be populated: {columns_with_nulls}. "
        "Review the missing-value summary above before continuing."
    )

print("No missing values found in the final feature dataset.")


,COLUMN,NULL_COUNT,NULL_PERCENTAGE
0,ARR_HOUR,0,0.0
1,CRS_DEP_TIME,0,0.0
2,DEP_HOUR,0,0.0
3,DEP_MINUTE,0,0.0
4,ORIGIN_STATE_NM,0,0.0
5,DEST_STATE_NM,0,0.0
6,ORIGIN_CITY_NAME,0,0.0
7,CRS_ELAPSED_TIME,0,0.0
8,DISTANCE,0,0.0
9,DEST,0,0.0


No missing values found in the final feature dataset.


In [13]:
# Review final feature schema

schema_rows = []

for field in df_ml.schema.fields:
    schema_rows.append(
        (
            field.name,
            field.dataType.simpleString(),
            field.nullable,
        )
    )

schema_df = spark.createDataFrame(
    schema_rows,
    [
        "COLUMN_NAME",
        "DATA_TYPE",
        "NULLABLE",
    ],
)

display(schema_df)


,COLUMN_NAME,DATA_TYPE,NULLABLE
0,QUARTER,int,True
1,MONTH,int,True
2,DAY_OF_WEEK,int,True
3,FL_DATE,date,True
4,OP_UNIQUE_CARRIER,string,True
5,OP_CARRIER_FL_NUM,int,True
6,ORIGIN,string,True
7,DEST,string,True
8,DISTANCE,double,True
9,ORIGIN_CITY_NAME,string,True


#### Save feature dataset

The validated feature dataset is stored as a managed Delta table within Unity Catalog.

The resulting table will be used as the input dataset for model training.


In [14]:
# Register feature dataset as managed Delta table

(
    df_ml.writeTo(cfg.FEATURES_TABLE)
    .using("delta")
    .createOrReplace()
)

print("Feature dataset saved successfully.")
print(f"Table: {cfg.FEATURES_TABLE}")
print(f"Rows: {spark.table(cfg.FEATURES_TABLE).count():,}")


Feature dataset saved successfully.
Table: workspace.default.flights_features
Rows: 6,879,483
